# Курсовой проект по ТОБД — загрузка и первичная проверка данных

**Тема:** Анализ данных о поездках такси для выявления факторов, влияющих на длительность поездок, и временных паттернов.

**Период:** январь 2024 года, Нью-Йорк.

В ноутбуке загружаются пять источников, проверяются первые пять записей, типы данных, основные статистики, пропуски и дубликаты. Первичная очистка **минимальная**: удаляются повторные записи и записи без ключевого идентификатора/времени. Пропуски в остальных полях **не заполняются и не удаляются**: подход к ним определим при дальнейшем анализе. Слияния источников, анализа факторов и создания новых признаков пока нет.

Исходные ответы XML/JSON остаются в `crashes_xml`, `weather_json`, `requests_311_json`. Для исследования формируются таблицы `crashes_df`, `weather_df`, `requests_311_df`. Реляционные источники — `taxi_df` и `taxi_zones_df`.

**Источники:** [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), [NYPD Collisions XML](https://data.cityofnewyork.us/d/h9gi-nx95), [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api), [NYC 311](https://data.cityofnewyork.us/d/erm2-nwe9), [Taxi Zone Lookup](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv).

**Зависимости:** `pandas`, `pyarrow`, `requests`. При необходимости: `%pip install pandas pyarrow requests`.

> Для ДТП и обращений исходные API-запросы ограничены `$limit=50000`. Это **не гарантия полноты** соответствующих источников за январь; перед исследованием временных паттернов проверим число записей и при необходимости добавим постраничную загрузку.

In [1]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET

## 1. NYC TLC — поездки жёлтого такси

Исходный Parquet за январь 2024 года. Одна запись соответствует поездке. Данные загружаются в `taxi_df` без каких-либо изменений.

In [2]:
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

taxi_df = pd.read_parquet(taxi_url)
print("NYC TLC:", taxi_df.shape)

NYC TLC: (2964624, 19)


## 2. NYPD — ДТП, XML

Официальный XML-ресурс NYC Open Data. В запросе выбран январь 2024 года. Исходный XML сохраняется **как дерево XML**, а не превращается в таблицу: `crashes_xml`.

Лимит 50 000 записей задан явно, потому что без него API возвращает только первую порцию данных. Если за месяц оказалось бы больше 50 000 ДТП, понадобилась бы отдельная постраничная загрузка.

In [3]:
crashes_url = "https://data.cityofnewyork.us/resource/h9gi-nx95.xml"
crashes_params = {
    "$where": "crash_date >= '2024-01-01T00:00:00' AND crash_date < '2024-02-01T00:00:00'",
    "$limit": 50000,
}

crashes_response = requests.get(crashes_url, params=crashes_params, timeout=120)
crashes_response.raise_for_status()
crashes_xml = ET.fromstring(crashes_response.content)
print("NYPD XML: документ загружен")

HTTPError: 403 Client Error: Forbidden for url: https://data.cityofnewyork.us/resource/h9gi-nx95.xml?%24where=crash_date+%3E%3D+%272024-01-01T00%3A00%3A00%27+AND+crash_date+%3C+%272024-02-01T00%3A00%3A00%27&%24limit=50000

## 3. Open-Meteo — архив погоды, JSON

Почасовые данные за январь 2024 года для координат центра Манхэттена. Ответ API сохраняется без преобразований как словарь Python `weather_json`.

In [4]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": 40.758,
    "longitude": -73.9855,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code",
    "timezone": "America/New_York",
}

weather_response = requests.get(weather_url, params=weather_params, timeout=120)
weather_response.raise_for_status()
weather_json = weather_response.json()
print("Open-Meteo JSON: документ загружен")

Open-Meteo JSON: документ загружен


## 4. NYC 311 — обращения о дорожных проблемах, JSON и текст

Для сопоставимости выбран тот же месяц. Из исходного датасета запрашиваются **только обращения категорий `Street Condition` и `Traffic Signal Condition`** (все доступные поля, в том числе исходные текстовые значения).

Ответ сохраняется как список исходных JSON-объектов `requests_311_json`, без удаления полей, нормализации текста и векторизации.

Лимит 50 000 — ограничение одной загрузки через SODA API 2.0. Если число подходящих обращений больше, дополнительные порции нужно будет загрузить отдельно; здесь это не скрывается.

In [5]:
requests_311_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
requests_311_params = {
    "$where": (
        "created_date >= '2024-01-01T00:00:00' AND "
        "created_date < '2024-02-01T00:00:00' AND "
        "complaint_type IN ('Street Condition', 'Traffic Signal Condition')"
    ),
    "$limit": 50000,
}

requests_311_response = requests.get(requests_311_url, params=requests_311_params, timeout=120)
requests_311_response.raise_for_status()
requests_311_json = requests_311_response.json()
print("NYC 311 JSON:", len(requests_311_json), "записей загружено")

HTTPError: 403 Client Error: Forbidden for url: https://data.cityofnewyork.us/resource/erm2-nwe9.json?%24where=created_date+%3E%3D+%272024-01-01T00%3A00%3A00%27+AND+created_date+%3C+%272024-02-01T00%3A00%3A00%27+AND+complaint_type+IN+%28%27Street+Condition%27%2C+%27Traffic+Signal+Condition%27%29&%24limit=50000

## 5. NYC TLC — справочник зон такси

CSV-справочник соответствия `LocationID` зонам и районам. Исходные столбцы и значения сохраняются как есть в `taxi_zones_df`.

In [6]:
taxi_zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

taxi_zones_df = pd.read_csv(taxi_zones_url)
print("Taxi zones:", taxi_zones_df.shape)

Taxi zones: (265, 4)


## 6. Приведение XML и JSON к табличному представлению

Это не слияние и не аналитическая обработка: преобразуем иерархические/JSON-ответы в таблицы для единообразных проверок. Необходимые числовые и временные поля получат корректные типы. Исходные `crashes_xml`, `weather_json`, `requests_311_json` не изменяются.

In [10]:
# NYPD XML: каждый внутренний <row> — одна запись о ДТП.
crash_records = crashes_xml.findall("./row/row")
crashes_df = pd.DataFrame([
    {field.tag: field.text for field in record}
    for record in crash_records
])

crash_numeric_cols = [
    "latitude", "longitude", "number_of_persons_injured", "number_of_persons_killed",
    "number_of_pedestrians_injured", "number_of_pedestrians_killed",
    "number_of_cyclist_injured", "number_of_cyclist_killed",
    "number_of_motorist_injured", "number_of_motorist_killed",
]
for column in crash_numeric_cols:
    if column in crashes_df.columns:
        crashes_df[column] = pd.to_numeric(crashes_df[column], errors="coerce")
crashes_df["crash_date"] = pd.to_datetime(crashes_df["crash_date"], errors="coerce")

# Open-Meteo JSON: hourly — набор параллельных почасовых массивов.
weather_df = pd.DataFrame(weather_json["hourly"])
weather_df["time"] = pd.to_datetime(weather_df["time"], errors="coerce")

# NYC 311 JSON: один словарь соответствует одному обращению.
requests_311_df = pd.DataFrame(requests_311_json)
requests_311_df["created_date"] = pd.to_datetime(
    requests_311_df["created_date"], errors="coerce"
)
if "closed_date" in requests_311_df.columns:
    requests_311_df["closed_date"] = pd.to_datetime(
        requests_311_df["closed_date"], errors="coerce"
    )

print("NYPD XML → DataFrame:", crashes_df.shape)
print("Open-Meteo JSON → DataFrame:", weather_df.shape)
print("NYC 311 JSON → DataFrame:", requests_311_df.shape)

NameError: name 'crashes_xml' is not defined

## 7. Проверка качества и минимальная очистка

Для каждого источника:

1. `head(5)` — первые пять строк;
2. `info()` — количество записей, поля, типы и непустые значения;
3. `describe(include="all")` — описательные статистики **числовых и категориальных** полей;
4. таблица количества и доли пропусков;
5. количество дубликатов (полных для TLC; по ключу для остальных);
6. удаление дублей и строк без **обязательного ключа**. Значения в остальных полях не трогаем.

Удаление дублей по ключу предполагает, что этот идентификатор уникален. Для TLC идентификатора отдельной поездки нет, поэтому удаляются только **полностью одинаковые строки**. Это не гарантирует, что все такие строки не являются отдельными поездками; число удалений выводится явно.

In [11]:
def inspect_and_clean(df, name, key=None, required=()):
    """Выводит базовый EDA и удаляет лишь дубликаты и строки без обязательных полей."""
    print(f"\n{'=' * 65}\n{name} | исходный размер: {df.shape}\n{'=' * 65}")

    print("\nПервые 5 строк:")
    display(df.head(5))

    print("\nСтруктура данных:")
    df.info()

    print("\nОписательные статистики:")
    display(df.describe(include="all").T)

    print("\nПропуски по столбцам:")
    missing = pd.DataFrame({
        "count": df.isna().sum(),
        "percent": (df.isna().mean() * 100).round(2),
    })
    display(missing)

    duplicates = df.duplicated(subset=key).sum()
    print(f"\nДубликатов {'по ключу ' + str(key) if key else 'полных строк'}: {duplicates:,}")

    missing_required = df[list(required)].isna().any(axis=1).sum() if required else 0
    print(f"Строк без обязательных полей {list(required)}: {missing_required:,}")

    cleaned = df.dropna(subset=list(required)).drop_duplicates(subset=key).reset_index(drop=True)
    print(f"После минимальной очистки: {cleaned.shape}; удалено строк: {len(df) - len(cleaned):,}")
    return cleaned

### 7.1. NYC TLC — поездки такси

Проверяем полные дубликаты. Для дальнейшего расчёта длительности поездки обязательны время посадки и время высадки. Остальные пропуски пока сохраняем.

In [12]:
taxi_df = inspect_and_clean(
    taxi_df,
    "NYC TLC — Yellow Taxi",
    required=("tpep_pickup_datetime", "tpep_dropoff_datetime"),
)


NYC TLC — Yellow Taxi | исходный размер: (2964624, 19)

Первые 5 строк:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0



Структура данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Air

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,2964624.0,NaN,NaN,NaN,1.754204,1.0,2.0,2.0,2.0,6.0,0.43259
tpep_pickup_datetime,2964624,NaN,NaN,NaN,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15,NaN
tpep_dropoff_datetime,2964624,NaN,NaN,NaN,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52,NaN
passenger_count,2824462.0,NaN,NaN,NaN,1.339281,0.0,1.0,1.0,1.0,9.0,0.850282
trip_distance,2964624.0,NaN,NaN,NaN,3.652169,0.0,1.0,1.68,3.11,312722.3,225.462572
RatecodeID,2824462.0,NaN,NaN,NaN,2.069359,1.0,1.0,1.0,1.0,99.0,9.823219
store_and_fwd_flag,2824462,2,N,2813126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,2964624.0,NaN,NaN,NaN,166.017884,1.0,132.0,162.0,234.0,265.0,63.623914
DOLocationID,2964624.0,NaN,NaN,NaN,165.116712,1.0,114.0,162.0,234.0,265.0,69.31535
payment_type,2964624.0,NaN,NaN,NaN,1.161271,0.0,1.0,1.0,1.0,4.0,0.580869



Пропуски по столбцам:


,count,percent
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,140162,4.73
trip_distance,0,0.00
RatecodeID,140162,4.73
store_and_fwd_flag,140162,4.73
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00



Дубликатов полных строк: 0
Строк без обязательных полей ['tpep_pickup_datetime', 'tpep_dropoff_datetime']: 0
После минимальной очистки: (2964624, 19); удалено строк: 0


### 7.2. NYPD — ДТП

Дубликаты определяем по `collision_id`. Для пространственно-временного анализа обязательны идентификатор ДТП и его дата. Отсутствие координат или числа пострадавших не является основанием удалять запись.

In [13]:
crashes_df = inspect_and_clean(
    crashes_df,
    "NYPD — Motor Vehicle Collisions",
    key=["collision_id"],
    required=("collision_id", "crash_date"),
)

NameError: name 'crashes_df' is not defined

### 7.3. Open-Meteo — погода

Уникальный ключ — почасовая временная метка `time`. Пропущенные метеоизмерения **не заменяем нулём**, поскольку отсутствие наблюдения не означает отсутствие осадков, ветра и т. п.

In [ ]:
weather_df = inspect_and_clean(
    weather_df,
    "Open-Meteo — Historical Weather",
    key=["time"],
    required=("time",),
)

### 7.4. NYC 311 — обращения

Уникальный идентификатор обращения — `unique_key`; временная характеристика — `created_date`. Текстовые и другие необязательные поля сохраняем без удаления или подмены. В этих записях могут встречаться вложенные структуры JSON; поэтому дубликаты определяем именно по ключу, а не по всем полям.

In [ ]:
requests_311_df = inspect_and_clean(
    requests_311_df,
    "NYC 311 — Street / Traffic Signal Conditions",
    key=["unique_key"],
    required=("unique_key", "created_date"),
)

### 7.5. NYC TLC — справочник зон

Уникальный ключ справочника — `LocationID`. Пропуски в названиях зон или районов сохраняем до этапа объединения.

In [ ]:
taxi_zones_df = inspect_and_clean(
    taxi_zones_df,
    "NYC TLC — Taxi Zones",
    key=["LocationID"],
    required=("LocationID",),
)

## 8. Итог — все пять таблиц доступны в ноутбуке

Для каждого источника подготовлена отдельная таблица. **Источники между собой не объединены.** Ни один необязательный пропуск не заполнялся фиктивным значением.

In [ ]:
datasets = {
    "taxi": taxi_df,
    "crashes": crashes_df,
    "weather": weather_df,
    "requests_311": requests_311_df,
    "taxi_zones": taxi_zones_df,
}

sizes = pd.DataFrame({
    name: {"rows": len(df), "columns": len(df.columns)}
    for name, df in datasets.items()
}).T
print("Размеры таблиц после первоначальной очистки:")
display(sizes)


---

**Результат этапа:** загрузка пяти источников, предварительный анализ их структуры, проверка качества и минимальная очистка. Далее необходимо отдельно исследовать физический смысл атрибутов и связи между источниками, определить границы допустимых значений и способы обработки оставшихся пропусков. Только после этого переходить к слиянию и анализу гипотезы.

**Ограничение воспроизводимости:** ответы API и результаты очистки появятся при реальном выполнении ноутбука в среде с доступом в интернет. Числа пропусков и дубликатов заранее не предполагаются.